<a href="https://www.kaggle.com/code/yassinekmimin/pfa-cv?scriptVersionId=286531893" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/luna16/sampleSubmission.csv
/kaggle/input/luna16/annotations.csv
/kaggle/input/luna16/candidates.csv
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.311236942972970815890902714604.mhd
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.212346425055214308006918165305.mhd
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.281967919138248195763602360723.mhd
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.707218743153927597786179232739.raw
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.217697417596902141600884006982.mhd
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.283569726884265181140892667131.raw
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.192256506776434538421891524301.raw
/kaggle/input/luna16/subset2/subset2/1.3.6.1.4.1.14519.5.2.1.6279.6001.113586291551175790743673929831.raw
/kaggle/input/luna16/subset2/subset2/

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("avc0706/luna16")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/luna16


# Import libraries

In [3]:
import os
import glob
import copy
import time

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm
from collections import namedtuple
import SimpleITK as sitk

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# Unify dataset

Annotations
The annotations file contains the center and diameter of each mass in CT scans.

In [4]:
df_annotations = pd.read_csv('/kaggle/input/luna16/annotations.csv')
df_annotations.head()

,seriesuid,coordX,coordY,coordZ,diameter_mm
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-128.699421,-175.319272,-298.387506,5.651471
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,103.783651,-211.925149,-227.121250,4.224708
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,69.639017,-140.944586,876.374496,5.786348
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,-24.013824,192.102405,-391.081276,8.143262
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,2.441547,172.464881,-405.493732,18.545150


In [5]:
df_annotations.shape

(1186, 5)

In [6]:
df_candidates = pd.read_csv('/kaggle/input/luna16/candidates_V2/candidates_V2.csv')
df_candidates.head()

,seriesuid,coordX,coordY,coordZ,class
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,68.420000,-74.480000,-288.700000,0
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-95.209361,-91.809406,-377.426350,0
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-24.766755,-120.379294,-273.361539,0
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,-63.080000,-65.740000,-344.240000,0
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,52.946688,-92.688873,-241.067872,0


In [7]:
df_candidates.shape

(754975, 5)

In [8]:
print(f'Total annotations: {df_annotations.shape[0]}, Unique CT scans: {len(df_annotations.seriesuid.unique())}')
print(f'Total candidates: {df_candidates.shape[0]}, Unique CT scans: {len(df_candidates.seriesuid.unique())}')

Total annotations: 1186, Unique CT scans: 601
Total candidates: 754975, Unique CT scans: 888


**Annotations:**

1,186 total annotations (nodules).

These annotations are spread across 601 unique CT scans (each CT scan can have multiple annotations).

**Candidates:**

754,975 total candidates (potential regions of interest).

These candidates are spread across 888 unique CT scans.

In [9]:
diameters = {}

# Loop through every annotation
for _, row in df_annotations.iterrows():
    
    # Create a tuple to represent the center
    center_xyz = (row.coordX, row.coordY, row.coordZ)
    
    # Append the center to the corresponding `seriesuid`
    diameters.setdefault(row.seriesuid, []).append(
        (center_xyz, row.diameter_mm)
    )

In [10]:
# Display first 3 CT scans and their annotations
for i, (uid, annots) in enumerate(diameters.items()):
    print(f"CT scan {i}: {uid}")
    for center, diameter in annots:
        print(f"  center={center}, diameter={diameter}")
    print("-" * 50)
    if i == 2:
        break


CT scan 0: 1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860
  center=(-128.6994211, -175.3192718, -298.3875064), diameter=5.651470635
  center=(103.7836509, -211.9251487, -227.12125), diameter=4.224708481
--------------------------------------------------
CT scan 1: 1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793540579077826395208
  center=(69.63901724, -140.9445859, 876.3744957), diameter=5.786347814
--------------------------------------------------
CT scan 2: 1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016233746780170740405
  center=(-24.0138242, 192.1024053, -391.0812764), diameter=8.143261683
  center=(2.441546798, 172.4648812, -405.4937318), diameter=18.54514997
  center=(90.93171321, 149.0272657, -426.5447146), diameter=18.20857028
  center=(89.54076865, 196.4051593, -515.0733216), diameter=16.38127631
--------------------------------------------------


In [11]:
%%time

# Using a namedtuple makes it easy to access values in a tuple
# using indexes or field names
CandidateInfoTuple = namedtuple(
    'CandidateInfoTuple',
    ['is_nodule', 'diameter_mm', 'series_uid', 'center_xyz']
)

# A list to store all candidates in the dataset
candidates = []

for _, row in df_candidates.iterrows():
    
    # Create a tuple to represent the candidate center
    # We suffix the name with `_xyz` to make it clear that we're using the
    # patient coordinate system: http://dicomiseasy.blogspot.com/2013/06/getting-oriented-using-image-plane.html
    candidate_center_xyz = (row.coordX, row.coordY, row.coordZ)

    # We begin by assuming the candidate doesn't have a corresponding annotation.
    # If this is the case, then the candidate will have a diameter of 0.
    candidate_diameter = 0.0
    
    # We then fetch the diameters of the CT scan we're looking at currently,
    # and loop over them to find a match for the candidate
    for annotation in diameters.get(row.seriesuid, []):
        
        # Extract the center and diameter of the annotation from the tuple
        annotation_center_xyz, annotation_diameter = annotation
        
        # For each of the coordinates - X, Y and Z, we check if
        # the candidate and the annotation are "close by"
        # (remember the really long and complicated sentence above?)
        
        # Since we've stored coordinates as tuples, we can index into them
        for i in range(3):
            
            # Find the absolute difference between the two coordinates
            delta = abs(candidate_center_xyz[i] - annotation_center_xyz[i])
            
            # If the coorindate of the candidate is more than half the radius away,
            # we don't consider it the same nodule as the annotation we're currently looking at
            if delta > annotation_diameter / 4:
                    break
            
        # The `else` block of a for loop in Python executes if the loop ends "naturally"
        # i.e. if it terminates because all iterations are complete, and not by a break statement
        # So if we go into this else block, then all 3 coordinates are within half the radius,
        # and we can consider the candidate and the annotation as the same nodule
        else:
            candidate_diameter = annotation_diameter
            
            # We don't need to look at any other remaining annotations,
            # because we've already found a match!
            break
            
            
    
    candidates.append(CandidateInfoTuple(
        bool(row['class']),
        candidate_diameter,
        row.seriesuid,
        candidate_center_xyz
    ))

CPU times: user 55.7 s, sys: 239 ms, total: 56 s
Wall time: 56 s


In [12]:
candidates.sort(reverse=True)

In [13]:
# Display first 5 unified candidates
for i, c in enumerate(candidates[:5]):
    print(f"Candidate {i}:")
    print(f"  is_nodule   : {c.is_nodule}")
    print(f"  diameter_mm : {c.diameter_mm}")
    print(f"  series_uid  : {c.series_uid}")
    print(f"  center_xyz  : {c.center_xyz}")
    print("-" * 50)

Candidate 0:
  is_nodule   : True
  diameter_mm : 32.27003025
  series_uid  : 1.3.6.1.4.1.14519.5.2.1.6279.6001.287966244644280690737019247886
  center_xyz  : (66.58022805, 82.56931698, -110.5421104)
--------------------------------------------------
Candidate 1:
  is_nodule   : True
  diameter_mm : 30.61040636
  series_uid  : 1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800
  center_xyz  : (48.23675256, 37.47721004, -98.64208784)
--------------------------------------------------
Candidate 2:
  is_nodule   : True
  diameter_mm : 30.61040636
  series_uid  : 1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800
  center_xyz  : (44.19, 37.79, -107.01)
--------------------------------------------------
Candidate 3:
  is_nodule   : True
  diameter_mm : 30.61040636
  series_uid  : 1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800
  center_xyz  : (40.69, 32.19, -97.15)
--------------------------------------------------
Candidate 4:
  is_nodule   : Tr

In [14]:
# Display first 5 TRUE nodules
count = 0
for c in candidates:
    if c.is_nodule:
        print(c)
        count += 1
        if count == 5:
            break


CandidateInfoTuple(is_nodule=True, diameter_mm=32.27003025, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.287966244644280690737019247886', center_xyz=(66.58022805, 82.56931698, -110.5421104))
CandidateInfoTuple(is_nodule=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(48.23675256, 37.47721004, -98.64208784))
CandidateInfoTuple(is_nodule=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(44.19, 37.79, -107.01))
CandidateInfoTuple(is_nodule=True, diameter_mm=30.61040636, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.112740418331256326754121315800', center_xyz=(40.69, 32.19, -97.15))
CandidateInfoTuple(is_nodule=True, diameter_mm=27.44242293, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.943403138251347598519939390311', center_xyz=(-45.29440163, 74.86925386, -97.52812481))


# Missing data

In [15]:
with open('/kaggle/input/missingcandidates/missing.txt', 'r') as f:
    missing_uids = {uid.split('\n')[0] for uid in f}
    
len(missing_uids)

443

There are 443 seriesuids in the annotations and candidates CSV files that don't have corresponding .mhd files.

We now remove from our list of candidates those that don't have a CT scan

In [16]:
candidates_clean = list(filter(lambda x: x.series_uid not in missing_uids, candidates))

print(f'All candidates in dataset: {len(candidates)}')
print(f'Candidates with CT scan  : {len(candidates_clean)}')

All candidates in dataset: 754975
Candidates with CT scan  : 377138


# Load the data


We'll now walk through how we want to convert the data we have into a format that we can consume with PyTorch.

We will walk through and understand all the steps using a single candidate before putting the code together into utility functions and a PyTorch Dataset.

In [17]:
candidate = candidates_clean[0]

candidate

CandidateInfoTuple(is_nodule=True, diameter_mm=32.27003025, series_uid='1.3.6.1.4.1.14519.5.2.1.6279.6001.287966244644280690737019247886', center_xyz=(66.58022805, 82.56931698, -110.5421104))

We use the glob module to find the .mhd and .raw files associated with the candidate.

The files could be in any one of the subset folders in the dataset. The glob module allows us to find the file by using patterns instead of manually looking inside each of the folders.

In [18]:
# Look for the file `<series_uid>.mhd` inside the `subset` folders
filepaths = glob.glob(f'/kaggle/input/luna16/subset*/*/{candidate.series_uid}.mhd')

# We removed all candidates that don't have corresponding CT scan files
# This line is another fail-safe to know when a CT scan doesn't exist
assert len(filepaths) != 0, f'CT scan with seriesuid {candidate.series_uid} not found!'

filepaths

['/kaggle/input/luna16/subset1/subset1/1.3.6.1.4.1.14519.5.2.1.6279.6001.287966244644280690737019247886.mhd']

This confirms that:

1.  The CT scan file exists on disk   
1.  Exactly one CT scan corresponds to this candidate
1. We now know where to load the 3D image from

In [19]:
mhd_file_path = filepaths[0]

mhd_file_path

'/kaggle/input/luna16/subset1/subset1/1.3.6.1.4.1.14519.5.2.1.6279.6001.287966244644280690737019247886.mhd'

In [20]:
mhd_file = sitk.ReadImage(mhd_file_path)

In [21]:
ct_scan = np.array(sitk.GetArrayFromImage(mhd_file), dtype=np.float32)

In [22]:
ct_scan.clip(-1000, 1000, ct_scan)

array([[[-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        ...,
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.]],

       [[-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        ...,
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.]],

       [[-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        [-1000., -1000., -1000., ..., -1000., -1000., -1000.],
        ...,
        [-10

In [23]:
origin_xyz = mhd_file.GetOrigin()
voxel_size_xyz = mhd_file.GetSpacing()
direction_matrix = np.array(mhd_file.GetDirection()).reshape(3, 3)

In [24]:
origin_xyz_np = np.array(origin_xyz)
voxel_size_xyz_np = np.array(voxel_size_xyz)

In [25]:
# Convert the coordinates of the center of the candidate
# from the patient coordinate system to column, row, index
cri = ((center_xyz - origin_xyz_np) @ np.linalg.inv(direction_matrix)) / voxel_size_xyz_np

# Since we'll be using column, row and index values to index into arrays,
# we round them to the nearest integer.
cri = np.round(cri)

# Going forward, we'll need the scan to be in the order index, row, column
# irc is the candidate center converted from patient (mm) coordinates
# to CT scan array indices (Index, Row, Column)

irc = (int(cri[2]), int(cri[1]), int(cri[0]))

In [26]:
ct_scan.shape

(123, 512, 512)

Since most of the CT scan doesn't contain any interesting to us, we will extract 3-dimensional chunks of the CT scan that contain nodules as input for our model.

Let's say we want to extract a chunk of size 10 along the index column, and 18 rows and columns.

In [27]:
dims_irc = (10, 18, 18)

In [28]:
# We will create three slices - one for each direction - to use to extract
# a region of interest from the CT scan
slice_list = []

for axis, center_val in enumerate(irc):
    
    # Get start and end index for the dimension so that the
    # nodule center is at the center of the 3d array we extract
    start_index = int(round(center_val - dims_irc[axis]/2))
    end_index = int(start_index + dims_irc[axis])

    # Adjust the indexes if the start_index is out of the CT scan array
    if start_index < 0:
        start_index = 0
        end_index = int(dims_irc[axis])
    
    # Do the same check for the end_index
    if end_index > ct_scan.shape[axis]:
        end_index = ct_scan.shape[axis]
        start_index = int(ct_scan.shape[axis] - dims_irc[axis])
        
    slice_list.append(slice(start_index, end_index))
    
tuple(slice_list)

(slice(68, 78, None), slice(288, 306, None), slice(223, 241, None))

We now have three slices we can use in each direction to extract the chunk we need.

The slice tuple defines a fixed-size 3D region of interest centered on the candidate, used to extract a (10,18,18) CT patch for model input.

In [29]:
ct_scan_chunk = ct_scan[tuple(slice_list)]
ct_scan_chunk.shape

(10, 18, 18)

We now have a CT scan chunk ct_scan_chunk. The center of the nodule is at index irc in the complete scan ct_scan.

The next step would be to convert this chunk of CT scan to a PyTorch tensor.

In [30]:
# Create a tensor from the NumPy array of the CT scan chunk
ct_scan_chunk_tensor = torch.from_numpy(ct_scan_chunk)

# convert it to a tensor of float32
ct_scan_chunk_tensor = ct_scan_chunk_tensor.to(torch.float32)
    
# Add an extra dimension to represent a single channel in the 3d image
ct_scan_chunk_tensor = ct_scan_chunk_tensor.unsqueeze(0)

ct_scan_chunk_tensor.shape

torch.Size([1, 10, 18, 18])

In [31]:
candidate.is_nodule

True

In [32]:
!pip install diskcache cassandra-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 8.0 MB/s eta 0:00:00


In [33]:
# The code in this cell is from the Deep Learning with PyTorch book's GitHub repository
# https://github.com/deep-learning-with-pytorch/dlwpt-code/blob/master/util/disk.py

# The imports have slightly been modified to make the code work


import gzip

from cassandra.cqltypes import BytesType
from diskcache import FanoutCache, Disk, core
from diskcache.core import io, MODE_BINARY
from io import BytesIO

class GzipDisk(Disk):
    def store(self, value, read, key=None):
        """
        Override from base class diskcache.Disk.

        Chunking is due to needing to work on pythons < 2.7.13:
        - Issue #27130: In the "zlib" module, fix handling of large buffers
          (typically 2 or 4 GiB).  Previously, inputs were limited to 2 GiB, and
          compression and decompression operations did not properly handle results of
          2 or 4 GiB.

        :param value: value to convert
        :param bool read: True when value is file-like object
        :return: (size, mode, filename, value) tuple for Cache table
        """
        # pylint: disable=unidiomatic-typecheck
        if type(value) is BytesType:
            if read:
                value = value.read()
                read = False

            str_io = BytesIO()
            gz_file = gzip.GzipFile(mode='wb', compresslevel=1, fileobj=str_io)

            for offset in range(0, len(value), 2**30):
                gz_file.write(value[offset:offset+2**30])
            gz_file.close()

            value = str_io.getvalue()

        return super(GzipDisk, self).store(value, read)


    def fetch(self, mode, filename, value, read):
        """
        Override from base class diskcache.Disk.

        Chunking is due to needing to work on pythons < 2.7.13:
        - Issue #27130: In the "zlib" module, fix handling of large buffers
          (typically 2 or 4 GiB).  Previously, inputs were limited to 2 GiB, and
          compression and decompression operations did not properly handle results of
          2 or 4 GiB.

        :param int mode: value mode raw, binary, text, or pickle
        :param str filename: filename of corresponding value
        :param value: database value
        :param bool read: when True, return an open file handle
        :return: corresponding Python value
        """
        value = super(GzipDisk, self).fetch(mode, filename, value, read)

        if mode == MODE_BINARY:
            str_io = BytesIO(value)
            gz_file = gzip.GzipFile(mode='rb', fileobj=str_io)
            read_csio = BytesIO()

            while True:
                uncompressed_data = gz_file.read(2**30)
                if uncompressed_data:
                    read_csio.write(uncompressed_data)
                else:
                    break

            value = read_csio.getvalue()

        return value

def getCache(scope_str):
    return FanoutCache('data-unversioned/cache/' + scope_str,
                       disk=GzipDisk,
                       shards=64,
                       timeout=1,
                       size_limit=3e11,
                       )

raw_cache = getCache('ct_scan_raw')

@raw_cache.memoize(typed=True)
def getCtScanChunk(series_uid, center_xyz, dims_irc):

        filepaths = glob.glob(f'/kaggle/input/luna16/subset*/*/{series_uid}.mhd')
        assert len(filepaths) != 0, f'CT scan with seriesuid {series_uid} not found!'
        mhd_file_path = filepaths[0]
        
        mhd_file = sitk.ReadImage(mhd_file_path)
        ct_scan = np.array(sitk.GetArrayFromImage(mhd_file), dtype=np.float32)
        ct_scan.clip(-1000, 1000, ct_scan)
        
        origin_xyz = mhd_file.GetOrigin()
        voxel_size_xyz = mhd_file.GetSpacing()
        direction_matrix = np.array(mhd_file.GetDirection()).reshape(3, 3)
        
        origin_xyz_np = np.array(origin_xyz)
        voxel_size_xyz_np = np.array(voxel_size_xyz)
        
        cri = ((center_xyz - origin_xyz_np) @ np.linalg.inv(direction_matrix)) / voxel_size_xyz_np
        cri = np.round(cri)
        irc = (int(cri[2]), int(cri[1]), int(cri[0]))
        
        slice_list = []
        for axis, center_val in enumerate(irc):
            
            start_index = int(round(center_val - dims_irc[axis]/2))
            end_index = int(start_index + dims_irc[axis])
            
            if start_index < 0:
                start_index = 0
                end_index = int(dims_irc[axis])
                
            if end_index > ct_scan.shape[axis]:
                end_index = ct_scan.shape[axis]
                start_index = int(ct_scan.shape[axis] - dims_irc[axis])

            slice_list.append(slice(start_index, end_index))
            
        ct_scan_chunk = ct_scan[tuple(slice_list)]
        
        return ct_scan_chunk

In [34]:
class LunaDataset(Dataset):
    
    def __init__(self, is_validation_set=False, validation_stride=0):
        '''Create a PyTorch dataset for the CT scans
        
        If `is_validation_set` is `True` then every `validation_stride` item is kept.
        Otherwise, every `validation_stride` item is deleted
        '''
        
        # Make a copy of all the candidates.
        # Pick every 350th candidate so that we have about 1k candidates in the dataset
        # It takes agonizingly long to load more data!
        self.candidates = copy.copy(candidates_clean[::350])
        
        # If this is the validation set, keep every `validation_stride` item
        if is_validation_set:
            self.candidates = self.candidates[::validation_stride]
        
        # If this is the training set, delete every `validation_stride` item
        else:
            del self.candidates[::validation_stride]
            
    def __len__(self):
        '''Returns the number of items in the dataset'''
        return len(self.candidates)
    
    def __getitem__(self, i):
        '''Get the `i`the item in the dataset'''
        
        # Get the `i`th candidate
        candidate = self.candidates[i]
        
        # We want to resize each CT scan to the following dimensions
        dims_irc = (10, 18, 18)
        
        # Use the utility function to fetch the CT scan
        ct_scan_np = getCtScanChunk(candidate.series_uid, candidate.center_xyz, dims_irc)
        
        # Convert the CT scan to a tensor
        ct_scan_tensor = torch.from_numpy(ct_scan_np).to(torch.float32).unsqueeze(0)
        
        # Convert the target to a tensor
        label_tensor = torch.tensor([
            not candidate.is_nodule,
            candidate.is_nodule
        ], dtype=torch.long)
        
        return ct_scan_tensor, label_tensor